In [1]:
import numpy as np
from typing import Callable
from dataclasses import dataclass
from edtrace import text, link, plot, image
from altair import Chart, Data
from einops import reduce
import functools
import tiktoken
import sys
sys.path.append('autumn2025-lectures')
from util import make_plot

In [2]:
def simple_binary_classifier(x: np.ndarray) -> int:
    logit = x[0] - x[1] - 1
    if logit > 0: 
        predicted_y = 1
    else:
        predicted_y = -1
    return predicted_y

In [3]:
x_a = np.array([1, 2])
predicted_y_a = simple_binary_classifier(x_a)
print(predicted_y_a)

-1


In [4]:
x_b = np.array([2, 0])
predicted_y_b = simple_binary_classifier(x_b)
print(predicted_y_b)

1


In [7]:
import numpy as np
import altair as alt
import pandas as pd
# Training data from the lecture
points = pd.DataFrame({
    'x0': [1, 2, 0],
    'x1': [2, 0, 0],
    'label': ['-1', '+1', '-1']
})
# Decision boundary: w · x + b = 0
# w = [1, -1], b = -1 → x0 - x1 - 1 = 0 → x1 = x0 - 1
boundary_x = np.linspace(-1, 4, 100)
boundary_y = boundary_x - 1
boundary = pd.DataFrame({'x0': boundary_x, 'x1': boundary_y})
# Scatter plot of example points
scatter = alt.Chart(points).mark_circle(size=150).encode(
    x='x0:Q',
    y='x1:Q',
    color=alt.Color('label:N', scale=alt.Scale(domain=['-1', '+1'], range=['blue', 'red']))
)
# Decision boundary line
line = alt.Chart(boundary).mark_line(color='black', strokeDash=[5, 5]).encode(
    x='x0:Q',
    y='x1:Q'
)
scatter + line

alt.LayerChart(...)

In [11]:
from deep_learning import Example


class Vocabulary:
    """Maps strings to integers."""
    def __init__(self):
        self.index_to_string: list[str] = []
        self.string_to_index: dict[str, int] = {}

    def get_index(self, string: str) -> int:
        index = self.string_to_index.get(string)
        if index is None:
            index = len(self.index_to_string)
            self.index_to_string.append(string)
            self.string_to_index[string] = index
        return index

    def get_string(self, index: int) -> str:
        return self.index_to_string[index]

    def __len__(self):
        return len(self.index_to_string)

    def asdict(self):
        return {
            "index_to_string": self.index_to_string,
            "string_to_index": self.string_to_index,
        }


def example_to_point(example: Example) -> dict:
    return {"x0": example.x[0], "x1": example.x[1], "color": "red" if example.target_y == 1 else "blue"}

In [12]:
def get_training_data():
    return [
        Example(x=np.array([1, 2]), target_y=-1),
        Example(x=np.array([2, 0]), target_y=1),
        Example(x=np.array([0, 0]), target_y=-1),
    ]

In [14]:
@dataclass(frozen=True)
class Example:
    x: np.ndarray
    target_y: float

In [ ]:
training_data = get_training_data()
data = [example_to_point(example) for example in trainging_data]
plot(make_plot("training data", "x0", "x1", f=None, points=data))

In [ ]:
from linear_classification import Parameters


def binary_classifier(params: Parameters, x: np.ndarray) -> float:
    """Applies the linear predictor given by `params` to input `x`."""
    logit = params.weight @ x + params.bias
    if logit > 0:
        predicted_y = 1
    else:
        predicted_y = -1
    return predicted_y

In [ ]:
from linear_classification import Parameters

params = Parameters(weight=np.array([1, -1]), bias=-1)
x = np.array([1, 1])
predicted_y = binary_classifier(params, x)
print(predicted_y)


-1


In [17]:
plot(make_plot("binary classifier", "x0", "x1", lambda x0: x0 + 1, points=[example_to_point(Example(x=x, target_y=predicted_y))]))

In [18]:
@dataclass(frozen=True)
class Parameters:
    weight: np.ndarray
    bias: float

In [20]:
params = Parameters(weight=np.array([1, -1,]), bias = 1)
x = np.array([1, 1])
predicted_y = binary_classifier(params, x)
print(predicted_y)

1


In [21]:
plot(make_plot("binary classifier", "x0", "x1", lambda x0: x0 + 1, points=[example_to_point(Example(x=x, target_y=predicted_y))]))

In [31]:
def squared_loss(example: Example, params: Parameters) -> float:
    predicted_y = example.x @ params.weight + params.bias
    residual = predicted_y - example.target_y
    loss = residual ** 2
    return loss

In [ ]:
params = Parameters(weight=np.array([1, -1]), bias = -1)
training_data = get_training_data()

loss = squared_loss(training_data[0], params)
print(loss)

1


In [24]:
plot(make_plot("squared loss", "residual", "loss", lambda residual: residual ** 2))

In [32]:
def zero_one_loss(example: Example, params: Parameters) -> float:
    predicted_y = binary_classifier(params, example.x)
    loss = int(predicted_y != example.target_y)
    return loss

In [ ]:
loss = zero_one_loss(Example(x=np.array([2, 0]), target_y=1), params)
print(loss)

0


In [26]:
loss = zero_one_loss(Example(x=np.array([0, -2]), target_y=-1), params)
print(loss)

1


In [33]:
def zero_one_loss_inline(example: Example, params: Parameters) -> float:
    logit = example.x @ params.weight + params.bias
    margin = logit * example.target_y
    loss = int(margin <= 0)
    return loss

In [27]:
from linear_classification import zero_one_loss_inline


loss = zero_one_loss_inline(Example(x=np.array([2, 0]), target_y=1), params)
print(loss)

0


In [28]:
plot(make_plot("zero-one loss", "margin", "loss", lambda margin: int(margin <= 0)))

In [34]:
def train_zero_one_loss(params: Parameters, training_data: list[Example]) -> float:
    losses = [zero_one_loss_inline(example, params) for example in training_data]
    train_loss = np.mean(losses)
    return train_loss

In [35]:
params = Parameters(weight=np.array([0, 0]), bias=-1)
train_loss = train_zero_one_loss(params, training_data)
print(train_loss)

0.3333333333333333


In [42]:
params = Parameters(weight=np.array([1, 1]), bias=0)
training_data = get_training_data()
train_loss = train_zero_one_loss(params, training_data)
print(train_loss)

0.6666666666666666


In [43]:
def gradient_zero_one_loss(example: Example, params: Parameters) -> Parameters:
    logit = example.x @ params.weight + params.bias
    margin = logit * example.target_y
    return Parameters(weight=np.zeros_like(params.weight), bias=0)

In [44]:
grad = gradient_zero_one_loss(training_data[0], params)
print(grad)

Parameters(weight=array([0, 0]), bias=0)


In [45]:
plot(make_plot("zero-one loss", "margin", "loss", lambda margin: int(margin <= 0)))

In [81]:
def logistic(logit: float) -> float:
    prob = 1 / (1 + np.exp(-logit))
    return prob

In [49]:
plot(make_plot("logistic function", "logit", "prob", logistic, xrange=(-10, 10)))

In [50]:
logit = 0
prob = logistic(logit)
print(prob)

0.5


In [51]:
logit = 1
prob = logistic(logit)
print(prob)

0.7310585786300049


In [52]:
logit = 8
prob = logistic(logit)
print(prob)

0.9996646498695336


In [53]:
logit = -1
prob = logistic(logit)
print(prob)

0.2689414213699951


In [54]:
logit = -8
prob = logistic(logit)
print(prob)

0.0003353501304664781


In [58]:
prob = 0.2
odds = prob / (1 - prob)
logit = np.log(odds)
prob2 = logistic(logit)
assert prob == prob2

print(prob2)

0.2


In [59]:
prob1 = logistic(logit=3)
prob2 = logistic(logit=-3)
print(prob1 + prob2)

1.0000000000000002


In [60]:
def gradient_logistic(logit: float) -> float:
    prob = logistic(logit)
    grad = prob * (1 - prob)
    return grad

In [62]:
grad_prob = gradient_logistic(logit=3)
print(grad_prob)


0.045176659730912


In [63]:
plot(make_plot("derivative of logistic function", "logit", "grad_prob", gradient_logistic, xrange=(-10, 10)))

In [64]:
params = Parameters(weight=np.array([1, -1]), bias = 1)
example = Example(x=np.array([2, 0]), target_y = 1)
predicted_y = binary_classifier(params, example.x)
print(predicted_y)

1


In [65]:
logit = example.x @ params.weight + params.bias
prob_pos = logistic(logit)
print(prob_pos)

0.9525741268224334


In [66]:
prob_neg = logistic(-logit)
print(prob_neg)

0.04742587317756678


In [67]:
margin = logit * example.target_y
prob_target = logistic(margin)
print(prob_target)

0.9525741268224334


In [68]:
log_prob_target = np.log(prob_target)
print(log_prob_target)

-0.04858735157374191


In [69]:
loss = -log_prob_target
print(loss)

0.04858735157374191


In [70]:
def logistic_loss(example: Example, params: Parameters) -> float:
    logit = example.x @ params.weight + params.bias
    margin = logit * example.target_y
    prob_target = logistic(margin)
    loss = -np.log(prob_target)
    return loss

In [71]:
loss = logistic_loss(example, params)
print(loss)

0.04858735157374191


In [73]:
data = make_plot("zero-one loss", "margin", "loss", lambda margin: int(margin <= 0))

plot(data)

In [74]:
data = make_plot("zero-one loss", "margin", "loss", lambda margin: int(margin <= 0))

plot(data)

In [75]:
training_data = get_training_data()

In [76]:
def train_logistic_loss(params: Parameters, training_data: list[Example]) -> float:
    losses = [logistic_loss(example, params) for example in training_data]
    train_loss = np.mean(losses)
    return train_loss

In [77]:
train_loss = train_logistic_loss(params, training_data)
print(train_loss)

0.68499873988397


In [83]:
def gradient_logistic_loss(example: Example, params: Parameters) -> Parameters:
    logit = example.x @ params.weight + params.bias
    margin = logit * example.target_y
    loss = -np.log(logistic(margin))
    grad_logit = -logistic(-margin)
    grad_weight = example.target_y * grad_logit * example.x
    grad_bias = example.target_y * grad_logit
    return Parameters(weight=grad_weight, bias=grad_bias)
  

In [79]:
def gradient_train_logistic_loss(params: Parameters, training_data: list[Example]) -> Parameters:
    grads = [gradient_logistic_loss(example, params) for example in training_data]
    mean_weight = np.mean([grad.weight for grad in grads], axis=0)
    mean_bias = np.mean([grad.bias for grad in grads])
    return Parameters(weight=mean_weight, bias=mean_bias)

In [84]:
params = Parameters(weight = np.array([0, 0]), bias=0)
example = Example(x=np.array([2, 0]), target_y=1)
grad = gradient_logistic_loss(example, params)
print(grad)

Parameters(weight=array([-1., -0.]), bias=np.float64(-0.5))


In [85]:
training_data = get_training_data()
grad = gradient_train_logistic_loss(params, training_data)
print(grad)

Parameters(weight=array([-0.16666667,  0.33333333]), bias=np.float64(0.16666666666666666))


In [86]:
training_data = get_training_data()
params = Parameters(weight=np.array([0, 0]), bias=0)
learning_rate = 1


In [87]:
losses = []
for step in range(20):
    train_loss = train_logistic_loss(params, training_data)
    grad = gradient_train_logistic_loss(params, training_data)
    params = Parameters(
        weight=params.weight - learning_rate * grad.weight,
        bias=params.bias - learning_rate * grad.bias,
    )
    losses.append(train_loss)

plot(Chart(Data(values=[{"step": i, "loss": loss} for i, loss in enumerate(losses)])).mark_line().encode(x="step:Q", y="loss:Q").to_dict())



In [88]:
points = [example_to_point(example) for example in training_data]

plot(make_plot("decision boundary", "x0", "x1", lambda x0: -(params.weight[0] * x0 + params.bias) / params.weight[1], points=points))